In [ ]:
def SNR_TH(bgsub_Signal, Background, n_pix, ron, darkCurrent, dit, ndit, screen_center, subarray_size):
    ron_per_pixel = ron/n_pix  # [e-/integration^(1/2)/pxl]
    darkCurrent_per_pixel = darkCurrent/n_pix # [e-/s/pxl]
    t = dit*ndit # [s]
    snr_th_map = t * bgsub_Signal / np.sqrt(t*bgsub_Signal + Background*t + t*darkCurrent_per_pixel + ndit*ron_per_pixel**2)
    
    # Mean of the snr_th on the subarray 
    limits_source = subarray_size
    signal_coords_implane = (screen_center-limits_source, 
                             screen_center+limits_source, 
                             screen_center-limits_source, 
                             screen_center+limits_source)
    
    snr_th_cropped_mean = np.nanmean(snr_th_map[signal_coords_implane[0]:signal_coords_implane[1], signal_coords_implane[2]:signal_coords_implane[3]])
    
    return snr_th_map, snr_th_cropped_mean

In [ ]:
def SNR_calculation(file, mode, chop_nod, exptime, SNR_implane_plot, SNR_readout_plot, subarray_size, distance_noise_subarray, psf_substraction, psf_file, object_name):
    """Determine the SNR 

    Args:
        file (FITS file): Input FITS file for METIS
        mode (string): either "img_n" or "img_lm"
        chop_nod (Boolean): True or False to perform (or not) the chop & nod
        exptime (Quantity): Time of exposition with unity following astropy package
        SNR_th (Boolean): True or False to perform (or not) the theoretical SNR (implane)
        SNR_empirical (Boolean): True or False to derive (or not) the Empirical SNR (readout)

    Returns:
        _type_: _description_
    """
    
    if mode == "img_n":
        band = "N-band"
    if mode == "img_lm":
        band = "L-band"
    
    
    # Sky observation
    cmd = sim.UserCommands(use_instrument="METIS", set_modes=[mode], properties={"!OBS.exptime": exptime.value, "!OBS.dit": None, "!OBS.ndit": None})
    metis = sim.OpticalTrain(cmd)
    metis.observe()      # blank sky
    
    sky_implane = metis.image_planes[0].data # [ph/s]
    sky_readout = metis.readout(exptime=exptime.value)[0]
    sky_readout_data = sky_readout[1].data
    dit, ndit = metis.cmds["!OBS.dit"], metis.cmds["!OBS.ndit"]
    
    # PSF substraction
    if psf_substraction == True:
        
        with fits.open(psf_file) as hdul:
            psf = sim.Source(image_hdu=hdul[0])
        metis.observe(psf)
        psf_implane = metis.image_planes[0].data # [ph/s]
        header_psf_implane = metis.image_planes[0].header
        psf_readout = metis.readout(edit=dit, ndit=ndit)[0]
        psf_readout_data = psf_readout[1].data
        header_psf_readout = psf_readout[1].header
        
        """# Le workflow pourrait ressembler à ceci - Herebelow psf should be "your psf" in the image plane. 
        evis_psf = EPSFStar(psf_file_unres, cutout_center=(1024, 1024))
        evis_psfs = EPSFStars([evis_psf]) 
        epsf_builder_euclid = EPSFBuilder(oversampling=4, maxiters=20, progress_bar=False) # , fitter=epsf_fitter)
        epsf_euclid, fitted_euclid = epsf_builder_euclid(evis_psfs) 
        psf_model_euclid = make_psf_model(epsf_euclid)"""
        
        
    # Object observation
    with fits.open(file) as hdul:
        src = sim.Source(image_hdu=hdul[0])
    metis.observe(src)

    src_implane = metis.image_planes[0].data # [ph/s]
    header_src_implane = metis.image_planes[0].header
    
    
    #Chop & nod
    if chop_nod == True:
        metis['chop_nod'].include = True
        print("Chopping:", metis.cmds['!OBS.chop_offsets'])
        print("Nodding: ", metis.cmds['!OBS.nod_offsets'])
        
        src_readout = metis.readout(dit=dit, ndit=ndit)[0]
        header_src_readout = src_readout[1].header
        src_readout_data = src_readout[1].data
        ndit *= 4 #due to chop & nod
        
        bgsub_readout = chop_nod_fct(src_readout, dit, ndit, mode)
        #sky_readout_data = sky_readout[1].data[screen_center-quadrant_size:screen_center+quadrant_size, screen_center-quadrant_size:screen_center+quadrant_size]
        
    if chop_nod == False:
        src_readout  = metis.readout(dit=dit, ndit=ndit)[0] 
        src_readout_data = src_readout[1].data
        screen_center = int(np.shape(src_readout_data)[0]/2)
        header_src_readout = src_readout[1].header
        


    # Background substraction 
    if psf_substraction == False:
        bgsub_implane = src_implane - sky_implane
        if chop_nod == False:
            bgsub_readout = src_readout_data - sky_readout_data #[ADU]
        
    if psf_substraction == True and chop_nod == False:
        bgsub_implane = src_implane - psf_implane
        bgsub_readout = src_readout_data - psf_readout_data #[ADU]
    
        
    """bgsub_implane = np.where(bgsub_implane > 0, bgsub_implane, 0) #in case the substraction gives neg nbr
    bgsub_readout = np.where(bgsub_readout > 0, bgsub_readout, 0) #in case the substraction gives neg nbr"""

    # Detector information 
    #metis.cmds["!DET"]
    n_pix = 2048**2 
    ron = metis.cmds["!DET.readout_noise"] # [e-/integration^(1/2)]
    darkCurrent = metis.cmds['!DET.dark_current'] # [e-/s]
    gain = metis.cmds["!DET.gain"] # [e-/ADU]
    if mode == "img_n":
        QE = 0.8 # [-]
    if mode == "img_lm":
        QE =  0.8574456 # [-]
        
        
    # Theoritical SNR & Mean of the snr_th on the subarray
    snr_th_map, snr_th_cropped_mean = SNR_TH(bgsub_implane*QE, sky_implane*QE, n_pix, ron, darkCurrent, dit, ndit, screen_center, subarray_size)
    
    
    
    # Estimation of SNR from the data
    snr_emp_map, snr_emp_mean, snr_th_cropped = SNR_DATA(bgsub_readout*gain, sky_readout_data*gain, dit, ndit, subarray_size, distance_noise_subarray, chop_nod)
    
    
    """"PLOTS"""
    if psf_substraction == True: 
        imsrcraw=plt.imshow(psf_implane, norm='log')
        plt.colorbar(imsrcraw, label='['+header_psf_implane['BUNIT']+']')
        plt.title('PSF Implane image '+band+' (with log norm)')
        plt.show()
        
        imsrcraw=plt.imshow(psf_readout_data, norm='log')
        plt.colorbar(imsrcraw, label='['+header_psf_readout['BUNIT']+']')
        plt.title('PSF readout image '+band+' (with log norm)')
        plt.show()
        
    if SNR_implane_plot == True or SNR_readout_plot == True:
        imsrcraw=plt.imshow(bgsub_implane, norm='log')
        plt.colorbar(imsrcraw, label='['+header_src_implane['BUNIT']+']')
        plt.title('Background substracted Implane image '+band+' (with log norm)')
        plt.show()

        imsrcraw=plt.imshow(bgsub_readout, norm='log')
        plt.colorbar(imsrcraw, label='['+header_src_readout['BUNIT']+']')
        plt.title('Background substracted readout image '+band+' (with log norm)')
        plt.show()
        
    if SNR_implane_plot == True:
        print(f"Maximum value of the theoretical SNR (implane): {np.max(snr_th_map):.2f}")
        print(f"Mean value of the theoretical SNR on the subarray (implane): {snr_th_cropped_mean:.2f}")
        
        plt.imshow(snr_th_map)
        plt.colorbar(label="SNR")
        plt.title("Theoretical SNR")
        plt.xlabel('pxl')
        plt.ylabel('pxl')
        plt.show()

    # TODO: change the ticks to correspond to the pixel number of the screen
    if SNR_readout_plot == True:
        print(f"Empirical SNR on the smaller subarray (readout): {snr_emp_mean:.2f}")
        print(f"Theoretical SNR on the smaller subarray (readout): {snr_th_cropped:.2f}")
        #Ratio
        print("Ratio theoritical SNR_implane and SNR_readout", snr_th_cropped_mean/snr_th_cropped)
        
        plt.imshow(snr_emp_map)
        plt.colorbar(label="SNR")
        plt.title("Empirical SNR (readout)")
        plt.xlabel('pxl')
        plt.ylabel('pxl')
        plt.show()
    
    """FITS file"""
    if SNR_implane_plot == True or SNR_readout_plot == True:
        # src
        create_fits(src_implane, header_src_implane, object_name=object_name+'_'+'src_implane', band_name=mode)
        create_fits(src_readout_data, header_src_readout, object_name=object_name+'_'+'src_readout', band_name=mode)
        
        # bgsub
        create_fits(bgsub_implane, header_src_implane, object_name=object_name+'_'+'bgsub_implane', band_name=mode)
        create_fits(bgsub_readout, header_src_readout, object_name=object_name+'_'+'bgsub_readout', band_name=mode)
        
        # snr 
        create_fits(snr_th_map, header_src_implane, object_name=object_name+'_'+'snr_th_map', band_name=mode)
        create_fits(snr_emp_map, header_src_readout, object_name=object_name+'_'+'snr_emp_map', band_name=mode)
        
        if psf_substraction == True: 
            # psf
            create_fits(psf_implane, header_psf_implane, object_name=object_name+'_'+'psf_implane', band_name=mode)
            create_fits(psf_readout_data, header_src_readout, object_name=object_name+'_'+'psf_readout', band_name=mode)
        
    return [snr_emp_mean,  snr_th_cropped, snr_th_cropped_mean, bgsub_implane, bgsub_readout, snr_th_map, snr_emp_map, src_readout_data, src_implane]

In [ ]:
cmd = sim.UserCommands(use_instrument="METIS", set_modes=["img_n"], properties={"!OBS.exptime": exptime_1hour.value, "!OBS.dit": None, "!OBS.ndit": None})
metis = sim.OpticalTrain(cmd)


om = metis.optics_manager
print("optics_manager type:", type(om))
print("dir(optics_manager) has:", [x for x in dir(om) if "effect" in x.lower() or "list" in x.lower()])

#['add_effect', 'all_effects', 'detector_array_effects', 'detector_effects', 'detector_setup_effects', 'fits_header_effects', 'fov_effects', 'fov_setup_effects', 'get_z_order_effects', 'image_plane_effects', 'image_plane_setup_effects', 'list_effects', 'load_effects', 'source_effects']
for attr in ["effects", "effect_objects", "effects_list", "all_effects"]:
    if hasattr(om, attr):
        effs = getattr(om, attr)
        print("Using", attr, "->", type(effs), "len=", len(effs))
        for i, e in enumerate(effs):
            print(i, e.__class__.__name__, getattr(e, "meta", {}).get("name",""))
        break


print(type(metis.effects))
print(metis.effects.colnames if hasattr(metis.effects, "colnames") else "no colnames")

psf_eff = metis.optics_manager.all_effects[12]   # from above #12
print(psf_eff)
print(psf_eff.meta)

dc = getattr(psf_eff, "data_container", None)
print("data_container:", dc)
print("filename:", getattr(dc, "filename", None))


#in terminal:
#(base) home@starlight ~ % find /Users/home/scopesim/inst_pkgs/METIS -type f \( -iname "*psf*" -o -iname "*kernel*" -o -iname "*pupil*" \)

#/Users/home/scopesim/inst_pkgs/METIS/PSF_N_9mag_06seeing.fits
#/Users/home/scopesim/inst_pkgs/METIS/PSF_SCAO_9mag_06seeing.fits
#/Users/home/scopesim/inst_pkgs/METIS/PSF_LM_9mag_06seeing.fits